# AOL Search Query Dataset Loader
This notebook downloads the AOL search session collection from Kaggle, aggregates raw search events into query-count pairs, and loads the data into a PostgreSQL database.

Credentials are loaded from the `.env` file. If the table already contains search queries, the download and aggregation stages are automatically skipped to avoid duplicate insertion.

In [17]:
# Install dependencies if needed
!pip install kagglehub pandas psycopg2-binary python-dotenv

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)


In [18]:
import os
import glob
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
import kagglehub
from dotenv import load_dotenv

# Load env variables from .env file
load_dotenv()

True

In [19]:
# Database configuration
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "typeahead_db")
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "18112006")

print("Connecting to default database to verify/create target database...")
try:
    conn_init = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname="postgres",
        user=DB_USER,
        password=DB_PASSWORD
    )
    conn_init.autocommit = True
    cur_init = conn_init.cursor()
    cur_init.execute(f"SELECT 1 FROM pg_catalog.pg_database WHERE datname = '{DB_NAME}'")
    exists = cur_init.fetchone()
    if not exists:
        print(f"Creating database {DB_NAME}...")
        cur_init.execute(f"CREATE DATABASE {DB_NAME}")
    else:
        print(f"Database {DB_NAME} already exists.")
    cur_init.close()
    conn_init.close()
except Exception as e:
    print(f"Error checking/creating database: {e}")

# Check if search_queries table exists and has rows
skip_data_load = False
try:
    conn = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )
    conn.autocommit = True
    cur = conn.cursor()
    
    # Create table if it doesn't exist yet
    cur.execute("""
        CREATE TABLE IF NOT EXISTS search_queries (
            id BIGSERIAL PRIMARY KEY,
            query VARCHAR(500) NOT NULL UNIQUE,
            count BIGINT NOT NULL DEFAULT 0
        );
    """)
    cur.execute("CREATE INDEX IF NOT EXISTS idx_search_queries_query ON search_queries (query);")
    
    # Check if table already has rows
    cur.execute("SELECT COUNT(*) FROM search_queries;")
    row_count = cur.fetchone()[0]
    if row_count > 0:
        print(f"Table 'search_queries' already has {row_count} rows. Data load will be skipped.")
        skip_data_load = True
    else:
        print("Table 'search_queries' is empty. Proceeding with data load.")
    cur.close()
    conn.close()
except Exception as e:
    print(f"Error verifying database table: {e}")

Connecting to default database to verify/create target database...
Database typeahead_db already exists.
Table 'search_queries' already has 1244495 rows. Data load will be skipped.


In [20]:
# Download the AOL user session collection dataset (500k queries)
if skip_data_load:
    print("Database already populated. Skipping download.")
else:
    print("Downloading dataset from Kaggle...")
    path = kagglehub.dataset_download("dineshydv/aol-user-session-collection-500k")
    print("Path to dataset files:", path)

Database already populated. Skipping download.


In [21]:
# Locate dataset log files
if skip_data_load:
    print("Database already populated. Skipping file discovery.")
else:
    all_files = glob.glob(os.path.join(path, "*.txt")) + glob.glob(os.path.join(path, "*.csv")) + glob.glob(os.path.join(path, "*.tsv"))
    print("Found files:", all_files)

Database already populated. Skipping file discovery.


In [22]:
# Read and combine all data
if skip_data_load:
    print("Database already populated. Skipping raw file loading.")
else:
    dfs = []
    for file in all_files:
        print(f"Reading {file}...")
        try:
            # AOL dataset files are usually tab-separated log files with a header
            df = pd.read_csv(file, sep='\t', header=0, on_bad_lines='skip', low_memory=False)
            dfs.append(df)
        except Exception as e:
            print(f"Error reading {file}: {e}")

    if dfs:
        combined_df = pd.concat(dfs, ignore_index=True)
        print(f"Combined DataFrame shape: {combined_df.shape}")
    else:
        raise ValueError("No data frames loaded!")

In [23]:
# Normalize column names and extract unique queries
if skip_data_load:
    print("Database already populated. Skipping aggregation logic.")
else:
    combined_df.columns = [c.strip().lower() for c in combined_df.columns]
    print("Columns found:", list(combined_df.columns))

    # Find query column
    query_col = None
    for col in combined_df.columns:
        if 'query' in col:
            query_col = col
            break

    if query_col is None:
        raise ValueError("Query column not found!")

    print(f"Using query column: '{query_col}'")

    # Clean queries: lowercase, strip, remove null/empty
    queries = combined_df[query_col].dropna().astype(str).str.strip().str.lower()
    queries = queries[queries != '']

    # Aggregate counts
    query_counts = queries.value_counts().reset_index()
    query_counts.columns = ['query', 'count']

    print(f"Total unique queries to insert: {len(query_counts)}")
    print("Sample queries:")
    print(query_counts.head(10))

Database already populated. Skipping raw file loading.


In [24]:
# Batch insert data into Postgres
if skip_data_load:
    print("Database already populated. Skipping data insertion.")
else:
    conn = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
)
    conn.autocommit = True
    cur = conn.cursor()
    
    data_tuples = list(query_counts.itertuples(index=False, name=None))

    insert_query = """
        INSERT INTO search_queries (query, count)
        VALUES %s
        ON CONFLICT (query) DO UPDATE
        SET count = search_queries.count + EXCLUDED.count
    """

    chunk_size = 10000
    print(f"Inserting {len(data_tuples)} records in chunks of {chunk_size}...")

    for i in range(0, len(data_tuples), chunk_size):
        chunk = data_tuples[i:i + chunk_size]
        execute_values(cur, insert_query, chunk)
        if (i // chunk_size) % 5 == 0 or (i + chunk_size) >= len(data_tuples):
            print(f"Inserted up to index {min(i + chunk_size, len(data_tuples))}...")

    print("PostgreSQL database population complete!")

    # Verify counts
    cur.execute("SELECT COUNT(*) FROM search_queries;")
    total_rows = cur.fetchone()[0]
    print(f"Total rows in table 'search_queries': {total_rows}")

    cur.close()
    conn.close()

Database already populated. Skipping aggregation logic.
